**REWARD DiD pipeline — engine v20.58.** Paths: one `INPUT_DIR` in `_paths.py` (output, temp, panel, results, estimator files and the shipped `REWARD_ground_inputs/` derive from it; RAM, GPU memory and every core are used up to 98 % of the total — `MEMORY_SHARE` there is an optional manual split, 1.0 = none). The design — timing (your fund workbook by default), dose, rings, years, seasons, fragments, overlaps, fixed effects, covariates — is set in each notebook's CELL 1 and applied when it runs (what CELL 1 leaves unset comes from P00's saved defaults). Multi-site: `08_Multisite_Runs/MS01`. Which models the data supports: P10. Pre-built packages: `python_prebuilt/` and the bundle's R routes (`R/lib`, see `R/README_BRIDGE.md`; the full R pipeline is the project RWDR). Details: the bundle's `docs/CHANGELOG_v20.md`, `PIPELINES_GUIDE.md`, `PREBUILT_MODEL_MAP.md`.

# V02 — Fixed-Effects Consistency Check
**Guards a REAL bug found in the pre-production audit.** Four modules were applying `Year`
alone as the time fixed effect while seventeen used `Year x Season`. With three seasons at very
different NDVI levels (Kharif ~0.55, Rabi ~0.35, Zaid ~0.12), and any imbalance in seasonal
coverage between treated and control pixels — routine in satellite panels — the Year-only spec
leaks seasonal variation into the treatment estimate.

**Measured impact: 3.4x more bias, 5.6% of the true effect size.**

This notebook fails loudly if a Year-only time FE is ever reintroduced.

In [ ]:
# ---- CELL 0: libraries (v17.3) -- run first; prints what to pip-install if anything is missing ----
import importlib as _il, importlib.util, sys as _sys
_REQ = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "scipy": "scipy", "sklearn": "scikit-learn",
        "psutil": "psutil", "tqdm": "tqdm", "joblib": "joblib", "openpyxl": "openpyxl"}
_OPT = {"torch": "GPU demeaning (CPU path without it)", "ipywidgets": "progress-bar widget (text bar without it)"}
_miss = [pipn for imp, pipn in _REQ.items() if _il.util.find_spec(imp) is None]
print("[INFO]    python", _sys.version.split()[0], "| missing REQUIRED:", _miss or "none",
      "| optional absent:", [k for k in _OPT if _il.util.find_spec(k) is None] or "none")
if _miss: print("[WARNING] run in a terminal:  pip install " + " ".join(_miss) + "   (v20.55: the engine installs a missing ESTIMATOR package itself -- wheel first, then source -- and confirms 'N of N installed' at every run; these base libraries it needs before it can start)")


In [ ]:
# ---- ROBUST IMPORT BOOTSTRAP (v9.0 fix for "ModuleNotFoundError: No module named '_common'") ----
# Jupyter's working directory is NOT always the notebook's folder, so a bare
# sys.path.insert(0,".") can fail. This searches the CWD, its parents, and their
# sub-folders for _common.py and reports exactly what it found.
import sys, os, glob as _glob

def _locate(fname, max_up=4):
    seen, cands = set(), []
    d = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if d in seen: break
        seen.add(d)
        cands.append(d)
        cands += [p for p in _glob.glob(os.path.join(d, "*")) if os.path.isdir(p)]
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    for c in cands:
        if os.path.exists(os.path.join(c, fname)):
            return c
    return None

_dir = _locate("_common.py")
if _dir is None:
    print("[FAILED]  Could not find _common.py")
    print(f"[INFO]    Jupyter's working directory is: {os.getcwd()}")
    print("[INFO]    Fix by ONE of these:")
    print("[INFO]      a) open this notebook from inside the folder that contains _common.py, OR")
    print("[INFO]      b) set the full path manually in the next line and re-run:")
    print("[INFO]         sys.path.insert(0, r'C:\\path\\to\\All_DIDsCodes')")
    raise ModuleNotFoundError("_common.py not found -- see the guidance above")
if _dir not in sys.path:
    sys.path.insert(0, _dir)
print(f"[OK]      found _common.py in: {_dir}")

sys.path.insert(0,"../All_DIDsCodes"); sys.path.insert(0,"All_DIDsCodes")
import numpy as np, pandas as pd
import _common as C
C.require_engine("20.44")     # minimum engine version; a newer engine is fine     # stop immediately on a mixed/cached engine
C.start_cell_log("V02")   # v20.36: every message of this notebook also goes to <OUTPUT_DIR>\cell_logs
C.autotune()          # v17: measures THIS machine (cores/RAM/VRAM/disk) and sets every performance knob
C.step(1,"simulate a panel with strong seasonality and UNBALANCED seasonal coverage")
rng=np.random.default_rng(4242); SE={"Kharif":.55,"Rabi":.35,"Zaid":.12}; TRUE=.06; rows=[]
for u in range(300):
    tr=1 if u<150 else 0; fe=rng.normal(0,.05)
    for yr in range(2018,2026):
        for s in ["Kharif","Rabi","Zaid"]:
            if rng.uniform()>(0.65 if (tr==1 and s=="Zaid") else 0.95): continue
            po=1 if yr>=2022 else 0
            rows.append({"pixel_id":u,"Year":yr,"time_fe_yearseason":f"{yr}_{s}",
                         "did_term":tr*po,"y":fe+SE[s]+.004*(yr-2018)+TRUE*tr*po+rng.normal(0,.02)})
d=pd.DataFrame(rows)
C.ok(f"{len(d):,} rows; treated pixels have deliberately sparser Zaid coverage")
C.done(1, next_task="next cell")


In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
C.step(2,"compare Year-only vs Year x Season time FE")
b1,_=C.estimate_twfe_did(d,"y","did_term","pixel_id","Year","pixel_id")
b2,_=C.estimate_twfe_did(d,"y","did_term","pixel_id","time_fe_yearseason","pixel_id")
e1,e2=abs(b1-TRUE),abs(b2-TRUE)
print(f"TRUE effect          : {TRUE:.5f}")
print(f"Year-only time FE    : {b1:.5f}   error {e1:.5f}")
print(f"Year x Season FE     : {b2:.5f}   error {e2:.5f}")
print(f"Year-only is {e1/e2:.1f}x more biased ({e1/TRUE*100:.1f}% of the true effect size)")
(C.ok if e2<e1 else C.fail)("Year x Season FE is less biased, as required")
assert e2<e1, "REGRESSION: a module may have reverted to a Year-only time FE"
C.ok("V02 COMPLETE -- season fixed effects are correctly absorbed")

# ---- MEMORY RELEASE (v14): free this model's frames so the next one starts clean ----
try:
    C.release(*[v for k,v in list(globals().items()) if k in ("panel","df","result") and v is not None])
except Exception as _e:
    print(f"[INFO]    release skipped: {_e}")
C.done(2, next_task="run V03_Classical_Estimator_Simulation_Validation.ipynb")